In [1]:
import os
# ✅ 1. 解決 oneDNN 衝突 (必留)
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

# ✅ 2. 穩定性開關：
# 如果你想聽 Claude 的試試 GPU 效能，請在下面這行前面加個 # 號
# 如果你的 Kernel 跑一跑又噴出 ExitCode: 3221225477 而死掉，請把這行改回來
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# ✅ 3. 繪圖與記憶體優化
plt.rcParams['figure.max_open_warning'] = 5
import matplotlib
matplotlib.use('Agg')  # 靜態後端，防止開啟過多視窗導致記憶體崩潰

# --- 1. 讀取數據 ---
# 請確保路徑與你的資料夾結構一致
df = pd.read_csv('ham10000/HAM10000_metadata.csv')

label_map = {
    'nv':    'Melanocytic nevus（痣）',
    'mel':   'Melanoma（黑色素瘤）',
    'bkl':   'Benign keratosis（良性角化）',
    'bcc':   'Basal cell carcinoma（基底細胞癌）',
    'akiec': 'Actinic keratosis（光化性角化）',
    'vasc':  'Vascular lesion（血管病變）',
    'df':    'Dermatofibroma（皮膚纖維瘤）',
}
df['label_name'] = df['dx'].map(label_map)

print("✅ 環境載入成功！")
print(f"目前 TensorFlow 版本: {tf.__version__}")
print(f"總資料筆數: {len(df)}")
print("\n各類別數量:")
print(df['dx'].value_counts())

# --- 2. 類別分布圖 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['dx'].value_counts()
colors = sns.color_palette("husl", 7)

axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title('Class Distribution (Count)', fontsize=14)
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=10)

axes[1].pie(counts.values, labels=counts.index, colors=colors, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Distribution (Percentage)')
plt.suptitle('HAM10000 — Severe Class Imbalance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 3. 圖片路徑對齊與展示 ---
img_dir_1 = r'C:\Users\chaon\Desktop\skin_cancer_analysis.ipynb\ham10000\HAM10000_images_part_1'

img_dir_2 = r'C:\Users\chaon\Desktop\skin_cancer_analysis.ipynb\ham10000\HAM10000_images_part_2'


def get_img_path(img_id):
    for d in [img_dir_1, img_dir_2]:
        p = os.path.join(d, f"{img_id}.jpg")
        if os.path.exists(p):
            return p
    return None

df['path'] = df['image_id'].apply(get_img_path)

# 檢查是否有遺漏圖片
missing = df['path'].isna().sum()
if missing > 0:
    print(f"⚠️ 警告: 有 {missing} 張圖片路徑找不到！")

# 展示樣本
classes = df['dx'].unique()
fig, axes = plt.subplots(7, 5, figsize=(15, 20))
for i, cls in enumerate(classes):
    samples = df[df['dx'] == cls].sample(5, random_state=42)
    for j, (_, row) in enumerate(samples.iterrows()):
        if row['path'] and os.path.exists(row['path']):
            img = Image.open(row['path'])
            img = img.resize((100, 75))  # 縮圖節省 RAM
            axes[i][j].imshow(img)
            img.close()
        axes[i][j].axis('off')
        if j == 0:
            axes[i][j].set_title(label_map[cls], fontsize=9, ha='left', x=0)

plt.suptitle('Sample Images per Class — HAM10000', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=80, bbox_inches='tight')
plt.close()

✅ 環境載入成功！
目前 TensorFlow 版本: 2.10.0
總資料筆數: 10015

各類別數量:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


C:\Users\chaon\AppData\Local\Temp\ipykernel_21680\1815028078.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\chaon\AppData\Local\Temp\ipykernel_21680\1815028078.py:97: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\chaon\AppData\Local\Temp\ipykernel_21680\1815028078.py:97: UserWarning: Glyph 33391 (\N{CJK UNIFIED IDEOGRAPH-826F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\chaon\AppData\Local\Temp\ipykernel_21680\1815028078.py:97: UserWarning: Glyph 24615 (\N{CJK UNIFIED IDEOGRAPH-6027}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\chaon\AppData\Local\Temp\ipykernel_21680\1815028078.py:97: UserWarning: Glyph 35282 (\N{CJK UNIFIED IDEOGRAPH-89D2}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\chaon\AppData\Local\Temp\ipykernel_21680\1815028078.py:97: UserWarning: Glyph 21270 (\N{CJK UNIFIED IDEOGRAPH-53

In [2]:
import subprocess
subprocess.run(['pip', 'install', 'tensorflow'], check=True)

CompletedProcess(args=['pip', 'install', 'tensorflow'], returncode=0)

In [3]:
# ============================================================
# STEP 2: 前處理 + Oversampling + Data Augmentation
# ============================================================
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

IMG_SIZE   = 224
BATCH_SIZE = 32

# ── Split ────────────────────────────────────────────────────
df_clean = df.dropna(subset=['path'])
print(f"有效圖片數量: {len(df_clean)}")

train_df, temp_df = train_test_split(df_clean, test_size=0.3,
                                     stratify=df_clean['dx'], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5,
                                     stratify=temp_df['dx'], random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── Oversampling：少數類別複製到與最多類別一樣多 ─────────────
max_count = train_df['dx'].value_counts().max()
oversampled = []
for cls in train_df['dx'].unique():
    cls_df = train_df[train_df['dx'] == cls]
    oversampled.append(cls_df.sample(max_count, replace=True, random_state=42))
train_df_balanced = pd.concat(oversampled).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nOversampling 後 train 數量: {len(train_df_balanced)}")
print("各類別數量（應該相等）:")
print(train_df_balanced['dx'].value_counts())

# ── class_weight 保留但訓練時不用 ───────────────────────────
classes_arr = np.array(sorted(df_clean['dx'].unique()))
weights     = compute_class_weight('balanced', classes=classes_arr, y=train_df['dx'])
class_weight_dict = {i: w for i, w in enumerate(weights)}

# ── ImageDataGenerator ───────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

# train_gen 用 oversampled 的 df
train_gen = train_datagen.flow_from_dataframe(
    train_df_balanced, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_gen = val_datagen.flow_from_dataframe(
    val_df, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_gen = val_datagen.flow_from_dataframe(
    test_df, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES  = len(CLASS_NAMES)
print("\nClass order:", CLASS_NAMES)

有效圖片數量: 10015
Train: 7010 | Val: 1502 | Test: 1503

Oversampling 後 train 數量: 32851
各類別數量（應該相等）:
dx
mel      4693
vasc     4693
df       4693
bkl      4693
nv       4693
bcc      4693
akiec    4693
Name: count, dtype: int64
Found 32851 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.

Class order: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [74]:
# ============================================================
# STEP 3A: MobileNetV2
# ============================================================
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

IMG_SIZE   = 224
BATCH_SIZE = 32

def build_mobilenet(num_classes=NUM_CLASSES):
    base = MobileNetV2(weights='imagenet', include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x   = base.output
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=base.input, outputs=out), base

mobilenet, mobilenet_base = build_mobilenet()

callbacks_m1 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6, monitor='val_loss'),
    ModelCheckpoint('best_mobilenet.h5', save_best_only=True,
                    monitor='val_accuracy', save_weights_only=True)
]

mobilenet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])

print("=== MobileNetV2 Phase 1 ===")
hist_m1 = mobilenet.fit(
    train_gen, validation_data=val_gen, epochs=15,
    callbacks=callbacks_m1          # 不加 class_weight
)
mobilenet.load_weights('best_mobilenet.h5')
print(f"Phase 1 最佳 val_accuracy: {max(hist_m1.history['val_accuracy']):.4f}")

# Phase 2
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False
for layer in mobilenet.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

callbacks_m2 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-6, monitor='val_loss'),
    ModelCheckpoint('best_mobilenet_ft.h5', save_best_only=True,
                    monitor='val_accuracy', save_weights_only=True)
]

mobilenet.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='categorical_crossentropy', metrics=['accuracy'])

print("\n=== MobileNetV2 Phase 2: Fine-tuning ===")
hist_m2 = mobilenet.fit(
    train_gen, validation_data=val_gen, epochs=15,
    callbacks=callbacks_m2          # 不加 class_weight
)
mobilenet.load_weights('best_mobilenet_ft.h5')
print(f"Phase 2 最佳 val_accuracy: {max(hist_m2.history['val_accuracy']):.4f}")
print("\nMobileNetV2 訓練完成！")

=== MobileNetV2 Phase 1 ===
Epoch 1/15
1027/1027 [==============================] - 321s 311ms/step - loss: 1.1678 - accuracy: 0.5710 - val_loss: 0.9119 - val_accuracy: 0.6525 - lr: 0.0010
Epoch 2/15
1027/1027 [==============================] - 307s 299ms/step - loss: 0.8847 - accuracy: 0.6695 - val_loss: 0.9660 - val_accuracy: 0.5919 - lr: 0.0010
Epoch 3/15
1027/1027 [==============================] - 304s 296ms/step - loss: 0.7779 - accuracy: 0.7062 - val_loss: 0.9720 - val_accuracy: 0.6205 - lr: 0.0010
Epoch 4/15
1027/1027 [==============================] - 304s 296ms/step - loss: 0.7193 - accuracy: 0.7316 - val_loss: 0.9416 - val_accuracy: 0.6411 - lr: 0.0010
Epoch 5/15
1027/1027 [==============================] - 304s 296ms/step - loss: 0.6149 - accuracy: 0.7709 - val_loss: 0.9078 - val_accuracy: 0.6491 - lr: 3.0000e-04
Epoch 6/15
1027/1027 [==============================] - 307s 299ms/step - loss: 0.5708 - accuracy: 0.7850 - val_loss: 0.9199 - val_accuracy: 0.6391 - lr: 3.0000e-0

In [7]:
# ============================================================
# STEP 3B: DenseNet121（取代 ResNet50）
# ============================================================
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

IMG_SIZE   = 224
BATCH_SIZE = 32

# ── 建立模型 ─────────────────────────────────────────────────
base = DenseNet121(weights='imagenet', include_top=False,
                   input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

x   = base.output
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dense(256, activation='relu')(x)
x   = layers.Dropout(0.4)(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
densenet = models.Model(base.input, out)

# 確認初始輸出平衡
test_out = densenet(tf.zeros([1, IMG_SIZE, IMG_SIZE, 3]), training=False)
print(f"初始輸出: {test_out.numpy()}")
print(f"最大值: {test_out.numpy().max():.4f}（理想接近 0.143）")

# ── Phase 1 ──────────────────────────────────────────────────
callbacks_d1 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, monitor='val_loss'),
    ModelCheckpoint('best_densenet.h5', save_best_only=True,
                    monitor='val_accuracy', save_weights_only=True)
]

densenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n=== DenseNet121 Phase 1 ===")
hist_r1 = densenet.fit(
    train_gen, validation_data=val_gen,
    epochs=15,
    steps_per_epoch=220,
    callbacks=callbacks_d1
)
densenet.load_weights('best_densenet.h5')
print(f"Phase 1 最佳 val_accuracy: {max(hist_r1.history['val_accuracy']):.4f}")

# ── Phase 2 ──────────────────────────────────────────────────
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print(f"Phase 2 可訓練層數: {sum(1 for l in densenet.layers if l.trainable)}")

callbacks_d2 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, monitor='val_loss'),
    ModelCheckpoint('best_densenet_ft.h5', save_best_only=True,
                    monitor='val_accuracy', save_weights_only=True)
]

densenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n=== DenseNet121 Phase 2: Fine-tuning ===")
hist_r2 = densenet.fit(
    train_gen, validation_data=val_gen,
    epochs=15,
    steps_per_epoch=220,
    callbacks=callbacks_d2
)
densenet.load_weights('best_densenet_ft.h5')
print(f"Phase 2 最佳 val_accuracy: {max(hist_r2.history['val_accuracy']):.4f}")
print("\nDenseNet121 訓練完成！")

29084464/29084464 [==============================] - 3s 0us/step
初始輸出: [[0.10293692 0.06663929 0.09387774 0.27858663 0.21780838 0.11959903
  0.12055204]]
最大值: 0.2786（理想接近 0.143）

=== DenseNet121 Phase 1 ===
Epoch 1/15
220/220 [==============================] - 99s 434ms/step - loss: 1.3870 - accuracy: 0.4850 - val_loss: 1.1338 - val_accuracy: 0.5952 - lr: 0.0010
Epoch 2/15
220/220 [==============================] - 96s 437ms/step - loss: 1.0817 - accuracy: 0.5909 - val_loss: 0.9999 - val_accuracy: 0.6119 - lr: 0.0010
Epoch 3/15
220/220 [==============================] - 67s 303ms/step - loss: 0.9635 - accuracy: 0.6375 - val_loss: 0.8933 - val_accuracy: 0.6538 - lr: 0.0010
Epoch 4/15
220/220 [==============================] - 85s 386ms/step - loss: 0.8907 - accuracy: 0.6663 - val_loss: 0.8273 - val_accuracy: 0.6518 - lr: 0.0010
Epoch 5/15
220/220 [==============================] - 66s 299ms/step - loss: 0.8432 - accuracy: 0.6875 - val_loss: 0.8912 - val_accuracy: 0.6545 - lr: 0.0010
Epo

In [76]:
# 診斷：確認資料流是否正常
import numpy as np

# 1. 確認 train_gen 類別分布
print("train_gen classes:", train_gen.class_indices)
print("val_gen classes:", val_gen.class_indices)

# 2. 取一個 batch 看看
train_gen.reset()
x_batch, y_batch = next(train_gen)
print(f"\ntrain batch shape: {x_batch.shape}")
print(f"pixel range: min={x_batch.min():.3f}, max={x_batch.max():.3f}")
print(f"label distribution: {y_batch.sum(axis=0).astype(int)}")

val_gen.reset()
x_val, y_val = next(val_gen)
print(f"\nval batch shape: {x_val.shape}")
print(f"pixel range: min={x_val.min():.3f}, max={x_val.max():.3f}")
print(f"label distribution: {y_val.sum(axis=0).astype(int)}")

# 3. 確認 class_weight_dict
print(f"\nclass_weight_dict: {class_weight_dict}")

# 4. 用隨機權重的模型預測一個 batch，確認 softmax 輸出正常
test_input = tf.zeros([1, IMG_SIZE, IMG_SIZE, 3])
test_out = resnet(test_input, training=False)
print(f"\nresnet output sum (should be ~1.0): {test_out.numpy().sum():.4f}")
print(f"resnet output: {test_out.numpy()}")

train_gen classes: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}
val_gen classes: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}

train batch shape: (32, 224, 224, 3)
pixel range: min=0.000, max=1.000
label distribution: [3 7 8 5 3 3 3]

val batch shape: (32, 224, 224, 3)
pixel range: min=0.000, max=1.000
label distribution: [ 0  3  5  0  5 18  1]

class_weight_dict: {0: 4.37305053025577, 1: 2.7817460317460316, 2: 1.3022478172023035, 3: 12.36331569664903, 4: 1.285530900421786, 5: 0.21338772031292808, 6: 10.115440115440116}

resnet output sum (should be ~1.0): 1.0000
resnet output: [[0.14334014 0.14341918 0.14190248 0.14148591 0.14395517 0.14489548
  0.14100161]]


In [10]:
# ============================================================
# STEP 3C: EfficientNetB3
# ============================================================
import os
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

IMG_SIZE   = 224
BATCH_SIZE = 32

eff_train_datagen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.15, horizontal_flip=True, vertical_flip=True,
    brightness_range=[0.8, 1.2], fill_mode='nearest'
)
eff_val_datagen = ImageDataGenerator()

# ← train_df_balanced（oversampled）
eff_train_gen = eff_train_datagen.flow_from_dataframe(
    train_df_balanced, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
eff_val_gen = eff_val_datagen.flow_from_dataframe(
    val_df, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
eff_test_gen = eff_val_datagen.flow_from_dataframe(
    test_df, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

def build_efficientnet(num_classes=NUM_CLASSES):
    base = EfficientNetB3(weights='imagenet', include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs), base

efficientnet, eff_base = build_efficientnet()

callbacks_eff = [
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, monitor='val_loss'),
    ModelCheckpoint('best_efficientnet.h5', save_best_only=True,
                    monitor='val_accuracy', save_weights_only=True)
]

efficientnet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                     loss='categorical_crossentropy', metrics=['accuracy'])

print("=== EfficientNetB3 Phase 1 ===")
hist_e1 = efficientnet.fit(
    eff_train_gen, validation_data=eff_val_gen, epochs=15,
    callbacks=callbacks_eff         # 不加 class_weight
)
efficientnet.load_weights('best_efficientnet.h5')
print(f"Phase 1 最佳 val_accuracy: {max(hist_e1.history['val_accuracy']):.4f}")

# Phase 2
def unfreeze_top_layers(base_model, num_layers=30):
    base_model.trainable = True
    for layer in base_model.layers[:-num_layers]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

unfreeze_top_layers(eff_base, num_layers=30)

efficientnet.compile(optimizer=tf.keras.optimizers.Adam(5e-5),
                     loss='categorical_crossentropy', metrics=['accuracy'])

callbacks_eff_p2 = [
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, monitor='val_loss'),
    ModelCheckpoint('best_efficientnet_ft.h5', save_best_only=True,
                    monitor='val_accuracy', save_weights_only=True)
]

print(f"解凍的可訓練層數: {sum(1 for l in eff_base.layers if l.trainable)}")
print("\n=== EfficientNetB3 Phase 2: Fine-tuning ===")
hist_e2 = efficientnet.fit(
    eff_train_gen, validation_data=eff_val_gen, epochs=15,
    callbacks=callbacks_eff_p2      # 不加 class_weight
)
efficientnet.load_weights('best_efficientnet_ft.h5')
print(f"Phase 2 最佳 val_accuracy: {max(hist_e2.history['val_accuracy']):.4f}")
print("\nEfficientNetB3 訓練完成！")

Found 32851 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.
=== EfficientNetB3 Phase 1 ===
Epoch 1/15
1027/1027 [==============================] - 1215s 1s/step - loss: 1.1400 - accuracy: 0.5735 - val_loss: 0.9108 - val_accuracy: 0.6531 - lr: 0.0010
Epoch 2/15
1027/1027 [==============================] - 1000s 972ms/step - loss: 0.8381 - accuracy: 0.6868 - val_loss: 0.8142 - val_accuracy: 0.6811 - lr: 0.0010
Epoch 3/15
1027/1027 [==============================] - 1101s 1s/step - loss: 0.7409 - accuracy: 0.7228 - val_loss: 0.8169 - val_accuracy: 0.6831 - lr: 0.0010
Epoch 4/15
1027/1027 [==============================] - 1335s 1s/step - loss: 0.6734 - accuracy: 0.7493 - val_loss: 0.9149 - val_accuracy: 0.6425 - lr: 0.0010
Epoch 5/15
1027/1027 [==============================] - 1246s 1s/step - loss: 0.6215 - accuracy: 0.7661 - val_loss: 0.8121 - val_accuracy: 0.6644 

In [4]:
# ============================================================
# STEP 3C-FL: EfficientNetB3 + Focal Loss
# ============================================================
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
import numpy as np
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

IMG_SIZE   = 224
BATCH_SIZE = 32

# ── Focal Loss ───────────────────────────────────────────────
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred  = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        ce      = -y_true * tf.math.log(y_pred)
        weight  = alpha * y_true * tf.math.pow(1.0 - y_pred, gamma)
        loss    = weight * ce
        return tf.reduce_sum(loss, axis=-1)
    return loss_fn

# ── EfficientNet 專用生成器（不 rescale）────────────────────
eff_train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
eff_val_datagen = ImageDataGenerator()

eff_train_gen_fl = eff_train_datagen.flow_from_dataframe(
    train_df_balanced, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
eff_val_gen_fl = eff_val_datagen.flow_from_dataframe(
    val_df, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# ── 建立模型 ─────────────────────────────────────────────────
def build_efficientnet_fl(num_classes=NUM_CLASSES):
    base = EfficientNetB3(weights='imagenet', include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs), base

efficientnet_fl, eff_base_fl = build_efficientnet_fl()

# ── Phase 1 ──────────────────────────────────────────────────
efficientnet_fl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=focal_loss(gamma=2.0),
    metrics=['accuracy']
)

callbacks_fl1 = [
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, monitor='val_loss'),
    ModelCheckpoint('best_efficientnet_fl.h5', save_best_only=True,
                    monitor='val_accuracy', save_weights_only=True)
]

print("=== EfficientNetB3 + Focal Loss Phase 1 ===")
hist_fl1 = efficientnet_fl.fit(
    eff_train_gen_fl, validation_data=eff_val_gen_fl,
    epochs=15, callbacks=callbacks_fl1
)
efficientnet_fl.load_weights('best_efficientnet_fl.h5')
print(f"Phase 1 最佳 val_accuracy: {max(hist_fl1.history['val_accuracy']):.4f}")

# ── Phase 2 ──────────────────────────────────────────────────
def unfreeze_top_layers(base_model, num_layers=30):
    base_model.trainable = True
    for layer in base_model.layers[:-num_layers]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

unfreeze_top_layers(eff_base_fl, num_layers=30)

efficientnet_fl.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss=focal_loss(gamma=2.0),
    metrics=['accuracy']
)

callbacks_fl2 = [
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.3, patience=3, min_lr=1e-7, monitor='val_loss'),
    ModelCheckpoint('best_efficientnet_fl_ft.h5', save_best_only=True,
                    monitor='val_accuracy', save_weights_only=True)
]

print(f"\n解凍層數: {sum(1 for l in eff_base_fl.layers if l.trainable)}")
print("=== EfficientNetB3 + Focal Loss Phase 2 ===")
hist_fl2 = efficientnet_fl.fit(
    eff_train_gen_fl, validation_data=eff_val_gen_fl,
    epochs=15, callbacks=callbacks_fl2
)
efficientnet_fl.load_weights('best_efficientnet_fl_ft.h5')
print(f"Phase 2 最佳 val_accuracy: {max(hist_fl2.history['val_accuracy']):.4f}")
print("\nEfficientNetB3 + Focal Loss 訓練完成！")

Found 32851 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
=== EfficientNetB3 + Focal Loss Phase 1 ===
Epoch 1/15
1027/1027 [==============================] - 313s 299ms/step - loss: 0.1849 - accuracy: 0.5586 - val_loss: 0.1233 - val_accuracy: 0.6318 - lr: 0.0010
Epoch 2/15
1027/1027 [==============================] - 290s 282ms/step - loss: 0.1241 - accuracy: 0.6596 - val_loss: 0.1182 - val_accuracy: 0.6025 - lr: 0.0010
Epoch 3/15
1027/1027 [==============================] - 286s 279ms/step - loss: 0.1048 - accuracy: 0.6997 - val_loss: 0.1199 - val_accuracy: 0.6411 - lr: 0.0010
Epoch 4/15
1027/1027 [==============================] - 295s 287ms/step - loss: 0.0951 - accuracy: 0.7192 - val_loss: 0.1116 - val_accuracy: 0.6212 - lr: 0.0010
Epoch 5/15
1027/1027 [==============================] - 296s 288ms/step - loss: 0.0866 - accuracy: 0.7375 - val_loss: 0.1104 - val_accuracy: 0.6172 - lr: 0.0010
Epoch 6/15
1027/1027 [======

In [5]:
# ============================================================
# Focal Loss 模型評估
# ============================================================
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 重建模型載入 weights
def build_efficientnet_fl_eval(num_classes=NUM_CLASSES):
    from tensorflow.keras.applications import EfficientNetB3
    base = EfficientNetB3(weights=None, include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

eff_fl_eval = build_efficientnet_fl_eval()
eff_fl_eval(tf.zeros([1, IMG_SIZE, IMG_SIZE, 3]), training=False)
eff_fl_eval.load_weights('best_efficientnet_fl_ft.h5')

# 用 eff_test_gen 評估
eff_test_gen_fl = ImageDataGenerator().flow_from_dataframe(
    test_df, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32, class_mode='categorical', shuffle=False
)

eff_test_gen_fl.reset()
fl_prob = eff_fl_eval.predict(eff_test_gen_fl, verbose=1)
fl_pred = np.argmax(fl_prob, axis=1)
y_true  = eff_test_gen_fl.classes

print("\n=== EfficientNetB3 + Focal Loss vs 原版 ===")
print(f"Focal Loss  — Accuracy: {accuracy_score(y_true, fl_pred)*100:.2f}%  "
      f"Macro F1: {f1_score(y_true, fl_pred, average='macro', zero_division=0)*100:.2f}%")
print(f"原版 CE     — Accuracy: 73.65%  Macro F1: 56.88%")
print("\n=== Per-class Report ===")
print(classification_report(y_true, fl_pred,
      target_names=list(eff_test_gen_fl.class_indices.keys()), zero_division=0))

Found 1503 validated image filenames belonging to 7 classes.
47/47 [==============================] - 11s 185ms/step

=== EfficientNetB3 + Focal Loss vs 原版 ===
Focal Loss  — Accuracy: 70.53%  Macro F1: 54.66%
原版 CE     — Accuracy: 73.65%  Macro F1: 56.88%

=== Per-class Report ===
              precision    recall  f1-score   support

       akiec       0.59      0.41      0.48        49
         bcc       0.59      0.64      0.61        77
         bkl       0.37      0.77      0.50       165
          df       0.47      0.47      0.47        17
         mel       0.43      0.39      0.41       167
          nv       0.93      0.77      0.84      1006
        vasc       0.37      0.82      0.51        22

    accuracy                           0.71      1503
   macro avg       0.54      0.61      0.55      1503
weighted avg       0.77      0.71      0.72      1503



In [6]:
# ============================================================
# STEP 4: 評估
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2, DenseNet121, EfficientNetB3
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve,
                             accuracy_score, f1_score)
from sklearn.preprocessing import label_binarize
 
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
 
assert 'test_df' in dir(), "❌ 請先執行 Step 2！"
 
IMG_SIZE   = 224
BATCH_SIZE = 32
 
# ── 重建 test_gen ─────────────────────────────────────────────
test_gen = ImageDataGenerator(rescale=1./255).flow_from_dataframe(
    test_df, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
eff_test_gen = ImageDataGenerator().flow_from_dataframe(
    test_df, x_col='path', y_col='dx',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
CLASS_NAMES = list(test_gen.class_indices.keys())
NUM_CLASSES  = len(CLASS_NAMES)
print(f"✅ 測試集重建完成: {CLASS_NAMES}")
 
# ── 重建架構（與各 Step 3 完全一致）─────────────────────────
def build_mobilenet_eval(num_classes=NUM_CLASSES):
    base = MobileNetV2(weights=None, include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-30]:   # Phase 2 解凍最後 30 層
        layer.trainable = False
    # MobileNet 訓練時沒有對 base.layers 做 BN freeze，這裡也不加
    x   = base.output
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=base.input, outputs=out)
 
def build_densenet_eval(num_classes=NUM_CLASSES):
    base = DenseNet121(weights=None, include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-30]:   # Phase 2 解凍最後 30 層
        layer.trainable = False
    for layer in base.layers:         # DenseNet 訓練時有凍結 BN
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    x   = base.output
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=base.input, outputs=out)
 
def build_efficientnet_eval(num_classes=NUM_CLASSES):
    base = EfficientNetB3(weights=None, include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-30]:   # Phase 2 解凍最後 30 層
        layer.trainable = False
    for layer in base.layers:         # EfficientNet 訓練時有凍結 BN
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)
 
# ── 載入權重 ──────────────────────────────────────────────────
def load_weights_safe(model, path):
    model(tf.zeros([1, IMG_SIZE, IMG_SIZE, 3]), training=False)
    model.load_weights(path)
    print(f"✅ {path} 載入成功 (共 {model.count_params():,} 參數)")
    return model
 
mobilenet    = load_weights_safe(build_mobilenet_eval(),    'best_mobilenet_ft.h5')
densenet     = load_weights_safe(build_densenet_eval(),     'best_densenet_ft.h5')
efficientnet = load_weights_safe(build_efficientnet_eval(), 'best_efficientnet_ft.h5')
 
# ── 評估函數 ──────────────────────────────────────────────────
def evaluate_model(model, generator, model_name):
    print(f"\n正在評估 {model_name}...")
    generator.reset()
    y_pred_prob = model.predict(generator, verbose=1)
    y_pred      = np.argmax(y_pred_prob, axis=1)
    y_true      = generator.classes
 
    print(f"\n{'='*50}\n  {model_name}\n{'='*50}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
 
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(f'confusion_{model_name}.png', dpi=150)
    plt.close()
 
    y_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
    plt.figure(figsize=(10, 7))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_prob[:, i])
        auc          = roc_auc_score(y_bin[:, i], y_pred_prob[:, i])
        plt.plot(fpr, tpr, label=f'{cls} (AUC={auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    plt.legend(loc='lower right', fontsize=9)
    plt.title(f'{model_name} — ROC Curves')
    plt.tight_layout()
    plt.savefig(f'roc_{model_name}.png', dpi=150)
    plt.close()
 
    return y_pred, y_pred_prob
 
# ── 執行評估 ──────────────────────────────────────────────────
m_pred, m_prob = evaluate_model(mobilenet,    test_gen,     'MobileNetV2')
d_pred, d_prob = evaluate_model(densenet,     test_gen,     'DenseNet121')
e_pred, e_prob = evaluate_model(efficientnet, eff_test_gen, 'EfficientNetB3')
 
# ── 訓練曲線 ──────────────────────────────────────────────────
def plot_history(hist_list, model_name):
    hist_list = [h for h in hist_list if h is not None]
    if not hist_list:
        print(f"找不到 {model_name} 訓練歷史，跳過。")
        return
    acc=[]; val_acc=[]; loss=[]; val_loss=[]
    for h in hist_list:
        acc      += h.history['accuracy']
        val_acc  += h.history['val_accuracy']
        loss     += h.history['loss']
        val_loss += h.history['val_loss']
 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(acc) + 1)
    ax1.plot(epochs, acc,     label='Train Acc',  color='#1E88E5')
    ax1.plot(epochs, val_acc, label='Val Acc',    color='#E53935', linestyle='--')
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch')
    ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, loss,     label='Train Loss', color='#1E88E5')
    ax2.plot(epochs, val_loss, label='Val Loss',   color='#E53935', linestyle='--')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch')
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.suptitle(model_name, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'training_curve_{model_name}.png', dpi=150)
    plt.close()
 
plot_history([hist_m1, hist_m2] if 'hist_m1' in dir() else [], 'MobileNetV2')
plot_history([hist_r1, hist_r2] if 'hist_r1' in dir() else [], 'DenseNet121')
plot_history([hist_e1, hist_e2] if 'hist_e1' in dir() else [], 'EfficientNetB3')
 
# ── 三模型比較表 ──────────────────────────────────────────────
y_true  = test_gen.classes
results = {}
for name, pred in [('MobileNetV2',   m_pred),
                   ('DenseNet121',    d_pred),
                   ('EfficientNetB3', e_pred)]:
    results[name] = {
        'Accuracy (%)':    round(accuracy_score(y_true, pred) * 100, 2),
        'Macro F1 (%)':    round(f1_score(y_true, pred, average='macro',    zero_division=0) * 100, 2),
        'Weighted F1 (%)': round(f1_score(y_true, pred, average='weighted', zero_division=0) * 100, 2),
    }
 
result_df = pd.DataFrame(results).T
print("\n=== 三模型比較 ===")
print(result_df.to_string())
 
ax = result_df.plot(kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='white')
plt.title('Model Performance Comparison')
plt.ylabel('Score (%)')
plt.ylim(0, 110)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
for p in ax.patches:
    ax.annotate(str(p.get_height()),
                (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='center', xytext=(0, 10),
                textcoords='offset points', fontsize=9)
plt.tight_layout()
plt.savefig('model_comparison_final.png', dpi=150)
plt.close()
 
print("\n✅ Step 4 完成！已儲存：")
print("  confusion_*.png / roc_*.png / training_curve_*.png / model_comparison_final.png")

Found 1503 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.
✅ 測試集重建完成: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
✅ best_mobilenet_ft.h5 載入成功 (共 2,624,839 參數)
✅ best_densenet_ft.h5 載入成功 (共 7,337,799 參數)
✅ best_efficientnet_ft.h5 載入成功 (共 11,216,950 參數)

正在評估 MobileNetV2...
47/47 [==============================] - 12s 233ms/step

  MobileNetV2
              precision    recall  f1-score   support

       akiec       0.50      0.37      0.42        49
         bcc       0.66      0.57      0.61        77
         bkl       0.42      0.70      0.52       165
          df       0.44      0.41      0.42        17
         mel       0.34      0.66      0.45       167
          nv       0.95      0.72      0.82      1006
        vasc       0.67      0.55      0.60        22

    accuracy                           0.69      1503
   macro avg       0.57      0.57      0.55      1503
weighted avg       0.78      0.69      0.71 

In [16]:
# ============================================================
# Ensemble: Soft Voting
# ============================================================
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 確認三個模型的 prob 都在
# m_prob, d_prob, e_prob 來自 Step 4
# fl_prob 來自剛才的 Focal Loss 評估

# ── 方案 A：三模型 ensemble（原版）──────────────────────────
ensemble_prob = (m_prob + d_prob + e_prob) / 3
ensemble_pred = np.argmax(ensemble_prob, axis=1)
y_true = test_gen.classes

print("=== Ensemble (MobileNet + DenseNet + EfficientNet) ===")
print(f"Accuracy: {accuracy_score(y_true, ensemble_pred)*100:.2f}%  "
      f"Macro F1: {f1_score(y_true, ensemble_pred, average='macro', zero_division=0)*100:.2f}%  "
      f"Weighted F1: {f1_score(y_true, ensemble_pred, average='weighted', zero_division=0)*100:.2f}%")

# ── 方案 B：加入 Focal Loss 版本的四模型 ensemble ───────────
# fl_prob 的 test_gen 順序跟 e_prob 一致（都是 eff_test_gen）
ensemble_prob_4 = (m_prob + d_prob + e_prob + fl_prob) / 4
ensemble_pred_4 = np.argmax(ensemble_prob_4, axis=1)

print("\n=== Ensemble (MobileNet + DenseNet + EfficientNet + FL) ===")
print(f"Accuracy: {accuracy_score(y_true, ensemble_pred_4)*100:.2f}%  "
      f"Macro F1: {f1_score(y_true, ensemble_pred_4, average='macro', zero_division=0)*100:.2f}%  "
      f"Weighted F1: {f1_score(y_true, ensemble_pred_4, average='weighted', zero_division=0)*100:.2f}%")

# ── 完整比較 ────────────────────────────────────────────────
print("\n=== 全部模型比較 ===")
results = {
    'MobileNetV2':        (accuracy_score(y_true, m_pred),    f1_score(y_true, m_pred,         average='macro', zero_division=0)),
    'DenseNet121':        (accuracy_score(y_true, d_pred),    f1_score(y_true, d_pred,         average='macro', zero_division=0)),
    'EfficientNetB3':     (accuracy_score(y_true, e_pred),    f1_score(y_true, e_pred,         average='macro', zero_division=0)),
    'EfficientNet+FL':    (accuracy_score(y_true, fl_pred),   f1_score(y_true, fl_pred,        average='macro', zero_division=0)),
    'Ensemble (3 model)': (accuracy_score(y_true, ensemble_pred),   f1_score(y_true, ensemble_pred,   average='macro', zero_division=0)),
    'Ensemble (4 model)': (accuracy_score(y_true, ensemble_pred_4), f1_score(y_true, ensemble_pred_4, average='macro', zero_division=0)),
}
print(f"{'Model':<25} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 47)
for name, (acc, f1) in results.items():
    print(f"{name:<25} {acc*100:>9.2f}% {f1*100:>9.2f}%")

=== Ensemble (MobileNet + DenseNet + EfficientNet) ===
Accuracy: 76.51%  Macro F1: 64.08%  Weighted F1: 77.95%

=== Ensemble (MobileNet + DenseNet + EfficientNet + FL) ===
Accuracy: 75.78%  Macro F1: 62.16%  Weighted F1: 77.06%

=== 全部模型比較 ===
Model                       Accuracy   Macro F1
-----------------------------------------------
MobileNetV2                   68.53%     54.97%
DenseNet121                   64.87%     51.70%
EfficientNetB3                73.65%     56.88%
EfficientNet+FL               70.53%     54.66%
Ensemble (3 model)            76.51%     64.08%
Ensemble (4 model)            75.78%     62.16%


In [12]:
# ============================================================
# Grad-CAM 完整修正版
# ============================================================
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array

IMG_SIZE = 224

def make_gradcam_heatmap(img_array, model, pred_index=None):
    eff_base = model.get_layer('efficientnetb3')
    
    grad_model = tf.keras.models.Model(
        inputs=eff_base.input,
        outputs=[
            eff_base.get_layer('top_activation').output,
            eff_base.output
        ]
    )
    
    with tf.GradientTape() as tape:
        conv_outputs, base_output = grad_model(img_array)
        x = model.get_layer('global_average_pooling2d_4')(base_output)
        x = model.get_layer('batch_normalization_4')(x, training=False)
        x = model.get_layer('dense_12')(x)
        x = model.get_layer('dropout_8')(x, training=False)
        x = model.get_layer('dense_13')(x)
        x = model.get_layer('dropout_9')(x, training=False)
        predictions = model.get_layer('dense_14')(x)
        
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads       = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = img_to_array(img).astype('uint8')

    jet          = matplotlib.colormaps['jet']
    jet_colors   = jet(np.arange(256))[:, :3]
    heatmap_uint = np.uint8(255 * heatmap)
    jet_heatmap  = jet_colors[heatmap_uint]
    jet_heatmap  = tf.keras.preprocessing.image.array_to_img(jet_heatmap * 255)
    jet_heatmap  = jet_heatmap.resize((IMG_SIZE, IMG_SIZE))
    jet_heatmap  = img_to_array(jet_heatmap)

    superimposed = jet_heatmap * alpha + img
    superimposed = tf.keras.preprocessing.image.array_to_img(superimposed)
    return img, superimposed

# ── 每個類別各取1張 ──────────────────────────────────────────
samples = {}
for cls in CLASS_NAMES:
    cls_files    = test_df[test_df['dx'] == cls]['path'].values
    samples[cls] = cls_files[0]

# ── 畫圖 ─────────────────────────────────────────────────────
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(12, 4 * len(CLASS_NAMES)))

for i, cls in enumerate(CLASS_NAMES):
    img_path = samples[cls]

    img              = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array        = img_to_array(img)
    img_array_exp    = np.expand_dims(img_array, axis=0)

    pred_prob = efficientnet(img_array_exp, training=False).numpy()[0]
    pred_cls  = CLASS_NAMES[np.argmax(pred_prob)]
    true_cls  = cls

    heatmap              = make_gradcam_heatmap(img_array_exp, efficientnet)
    original, superimposed = overlay_gradcam(img_path, heatmap)

    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f'True: {true_cls}', fontsize=10)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(np.uint8(255 * heatmap), cmap='jet')
    axes[i, 1].set_title('Heatmap', fontsize=10)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(superimposed)
    correct = 'CORRECT' if pred_cls == true_cls else 'WRONG'
    axes[i, 2].set_title(
        f'Pred: {pred_cls} [{correct}]\nConf: {pred_prob.max()*100:.1f}%',
        fontsize=10,
        color='green' if correct == 'CORRECT' else 'red'
    )
    axes[i, 2].axis('off')

plt.suptitle('EfficientNetB3 — Grad-CAM Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_efficientnet.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grad-CAM 儲存完成：gradcam_efficientnet.png")

✅ Grad-CAM 儲存完成：gradcam_efficientnet.png


C:\Users\chaon\AppData\Local\Temp\ipykernel_21680\2304818329.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ============================================================
# STEP 5: Grad-CAM 可解釋熱力圖（DenseNet121）
# ============================================================
import subprocess
subprocess.run(['pip', 'install', 'opencv-python-headless', '-q'], check=True)
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import os
 
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
 
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    try:
        grad_model = tf.keras.models.Model(
            inputs=model.inputs,
            outputs=[model.get_layer(last_conv_layer_name).output, model.output]
        )
    except ValueError:
        print(f"❌ 找不到層名稱: {last_conv_layer_name}，請檢查 model.summary()")
        return None
 
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
 
    grads       = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()
 
def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img     = cv2.imread(img_path)
    img     = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlaid = cv2.addWeighted(img, 1 - alpha, heatmap, alpha, 0)
    return cv2.cvtColor(overlaid, cv2.COLOR_BGR2RGB)
 
# DenseNet121 最後卷積層名稱
LAST_CONV = 'conv5_block16_concat'
 
if 'test_df' not in dir():
    print("❌ 錯誤：找不到 test_df，請先執行 Step 2！")
else:
    num_samples_per_class = 2
    fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(16, NUM_CLASSES * 3))
    fig.suptitle('Grad-CAM Visualization — DenseNet121\n(Heatmap shows model focus regions for diagnosis)',
             fontsize=16, fontweight='bold')
 
    for i, cls in enumerate(CLASS_NAMES):
        samples = test_df[test_df['dx'] == cls].sample(
            min(num_samples_per_class, len(test_df[test_df['dx'] == cls])),
            random_state=42
        )
 
        for idx, (_, row) in enumerate(samples.iterrows()):
            img_path  = row['path']
            img_orig  = Image.open(img_path).resize((IMG_SIZE, IMG_SIZE))
            img_array = np.expand_dims(np.array(img_orig) / 255.0, axis=0).astype('float32')
 
            heatmap = get_gradcam_heatmap(densenet, img_array, LAST_CONV)
 
            if heatmap is not None:
                overlaid = overlay_gradcam(img_path, heatmap)
                axes[i][idx*2].imshow(img_orig)
                axes[i][idx*2].set_title(f"True: {cls}\nOriginal", fontsize=9)
                axes[i][idx*2].axis('off')
                axes[i][idx*2+1].imshow(overlaid)
                axes[i][idx*2+1].set_title("Grad-CAM", fontsize=9)
                axes[i][idx*2+1].axis('off')
            else:
                axes[i][idx*2].text(0.5, 0.5, 'Layer Name Error', ha='center')
                axes[i][idx*2].axis('off')
 
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('gradcam_results.png', dpi=120)
    plt.close()
    print("✅ Step 5 完成！已儲存：gradcam_results.png")

✅ Step 5 完成！已儲存：gradcam_results.png


In [29]:
# ============================================================
# STEP 5B: Grad-CAM — EfficientNetB3 修正版2
# ============================================================
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB3
from PIL import Image
import matplotlib.pyplot as plt
import os

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
IMG_SIZE = 224

# ── 用跟訓練完全一樣的架構載入 ──────────────────────────────
def build_efficientnet_eval_gc(num_classes=NUM_CLASSES):
    base = EfficientNetB3(weights=None, include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs), base

eff_gc_model, eff_gc_base = build_efficientnet_eval_gc()
eff_gc_model(tf.zeros([1, IMG_SIZE, IMG_SIZE, 3]), training=False)
eff_gc_model.load_weights('best_efficientnet_ft.h5')
print("✅ EfficientNetB3 載入成功")

# ── last conv 從 base submodel 裡取 ─────────────────────────
conv_layers = [l.name for l in eff_gc_base.layers if 'conv' in l.name.lower()]
LAST_CONV = conv_layers[-1]
print(f"使用 last conv layer: {LAST_CONV}")

# ── Grad-CAM：對 base submodel 建 grad_model ────────────────
def get_gradcam_heatmap(outer_model, base_model, img_array, last_conv_name):
    grad_model_base = tf.keras.models.Model(
        inputs=base_model.inputs,
        outputs=[base_model.get_layer(last_conv_name).output,
                 base_model.output]
    )
    
    # 找出 outer_model 裡 base 之後的層
    post_base_input = tf.keras.Input(shape=grad_model_base.output[1].shape[1:])
    px = post_base_input
    found = False
    for layer in outer_model.layers:
        if found:
            px = layer(px)
        if layer.name == base_model.name:
            found = True
    post_base_model = tf.keras.Model(inputs=post_base_input, outputs=px)

    with tf.GradientTape() as tape:
        conv_out, base_out = grad_model_base(img_array)
        tape.watch(conv_out)
        preds       = post_base_model(base_out)
        pred_index  = tf.argmax(preds[0])
        class_score = preds[:, pred_index]

    grads        = tape.gradient(class_score, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out_val = conv_out[0]
    heatmap      = conv_out_val @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img     = cv2.imread(img_path)
    img     = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    overlaid = cv2.addWeighted(img, 1 - alpha, heatmap, alpha, 0)
    return cv2.cvtColor(overlaid, cv2.COLOR_BGR2RGB)

CLASS_NAMES_EVAL = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(16, NUM_CLASSES * 3))
fig.suptitle('Grad-CAM Visualization — EfficientNetB3\n(Heatmap shows model focus regions for diagnosis)',
             fontsize=16, fontweight='bold')

for i, cls in enumerate(CLASS_NAMES_EVAL):
    samples = test_df[test_df['dx'] == cls].sample(
        min(2, len(test_df[test_df['dx'] == cls])), random_state=42
    )
    for idx, (_, row) in enumerate(samples.iterrows()):
        img_path  = row['path']
        img_orig  = Image.open(img_path).resize((IMG_SIZE, IMG_SIZE))
        img_array = np.expand_dims(np.array(img_orig), axis=0).astype('float32')

        try:
            heatmap  = get_gradcam_heatmap(eff_gc_model, eff_gc_base, img_array, LAST_CONV)
            overlaid = overlay_gradcam(img_path, heatmap)
            axes[i][idx*2].imshow(img_orig)
            axes[i][idx*2].set_title(f"True: {cls}\nOriginal", fontsize=9)
            axes[i][idx*2].axis('off')
            axes[i][idx*2+1].imshow(overlaid)
            axes[i][idx*2+1].set_title("Grad-CAM", fontsize=9)
            axes[i][idx*2+1].axis('off')
        except Exception as e:
            print(f"❌ {cls} idx{idx}: {e}")
            axes[i][idx*2].axis('off')
            axes[i][idx*2+1].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('gradcam_efficientnet.png', dpi=120)
plt.close()
print("✅ 已儲存 gradcam_efficientnet.png")

✅ EfficientNetB3 載入成功
使用 last conv layer: top_conv
✅ 已儲存 gradcam_efficientnet.png


In [20]:
# ============================================================
# STEP 6: 三模型比較總表
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
import os

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

y_true  = test_gen.classes
results = {}
for name, pred in [('MobileNetV2',   m_pred),
                   ('DenseNet121',    d_pred),
                   ('EfficientNetB3', e_pred)]:
    results[name] = {
        'Accuracy (%)':    round(accuracy_score(y_true, pred) * 100, 2),
        'Macro F1 (%)':    round(f1_score(y_true, pred, average='macro',    zero_division=0) * 100, 2),
        'Weighted F1 (%)': round(f1_score(y_true, pred, average='weighted', zero_division=0) * 100, 2),
    }

result_df = pd.DataFrame(results).T
print("\n=== 各模型性能比較總表 ===")
print(result_df.to_string())

ax = result_df.plot(kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='white')
plt.title('Model Performance Comparison (Accuracy vs F1)', fontsize=14, fontweight='bold')
plt.ylabel('Score (%)')
plt.ylim(0, 110)
plt.xticks(rotation=0)
plt.legend(loc='lower right', frameon=True, shadow=True)
plt.grid(axis='y', alpha=0.3, linestyle='--')
for p in ax.patches:
    ax.annotate(str(p.get_height()),
                (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='center', xytext=(0, 10),
                textcoords='offset points', fontsize=9)
plt.tight_layout()
plt.savefig('model_comparison_final.png', dpi=150)
plt.close()
print("✅ Step 6 完成！")


=== 各模型性能比較總表 ===
                Accuracy (%)  Macro F1 (%)  Weighted F1 (%)
MobileNetV2            68.53         54.97            71.26
DenseNet121            64.87         51.70            68.23
EfficientNetB3         73.65         56.88            74.78
✅ Step 6 完成！


In [27]:
# 修正：固定用 7 類載入（跟訓練時一致）
TRAIN_CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']  # 訓練時的順序

def build_mobilenet_real():
    base = MobileNetV2(weights=None, include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    x   = base.output
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(7, activation='softmax')(x)  # ← 固定 7
    return models.Model(inputs=base.input, outputs=out)

def build_densenet_real():
    base = DenseNet121(weights=None, include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    x   = base.output
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(7, activation='softmax')(x)  # ← 固定 7
    return models.Model(inputs=base.input, outputs=out)

def build_efficientnet_real():
    base = EfficientNetB3(weights=None, include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(7, activation='softmax')(x)  # ← 固定 7
    return models.Model(inputs, outputs)

mn_real  = load_w(build_mobilenet_real(),    'best_mobilenet_ft.h5')
dn_real  = load_w(build_densenet_real(),     'best_densenet_ft.h5')
eff_real = load_w(build_efficientnet_real(), 'best_efficientnet_ft.h5')

# 預測
print("\n預測中...")
real_gen.reset();      mn_prob_r  = mn_real.predict(real_gen,      verbose=1)
real_gen.reset();      dn_prob_r  = dn_real.predict(real_gen,      verbose=1)
real_gen_eff.reset();  eff_prob_r = eff_real.predict(real_gen_eff, verbose=1)

# Ensemble（7類機率）
ensemble_prob_r = (mn_prob_r + dn_prob_r + eff_prob_r) / 3

# 把 7 類預測對應到 6 類 ground truth
# TRAIN_CLASS_NAMES 的 index → 類別名稱 → 對應到 real_gen 的 index
real_class_indices = real_gen.class_indices  # {'bcc':0, 'bkl':1, 'df':2, 'mel':3, 'nv':4, 'vasc':5}

# 重新對應：只取 real_gen 有的 6 類機率
keep_indices = []
keep_names   = []
for i, cls in enumerate(TRAIN_CLASS_NAMES):
    if cls in real_class_indices:
        keep_indices.append(i)
        keep_names.append(cls)

# 取出 6 類的機率，重新 argmax
filtered_prob = ensemble_prob_r[:, keep_indices]
ensemble_pred_r = np.argmax(filtered_prob, axis=1)
y_true_r = real_gen.classes

print("\n=== Real Test Results (ISIC 2019 — 6 classes) ===")
print(f"Accuracy:    {accuracy_score(y_true_r, ensemble_pred_r)*100:.2f}%")
print(f"Macro F1:    {f1_score(y_true_r, ensemble_pred_r, average='macro',    zero_division=0)*100:.2f}%")
print(f"Weighted F1: {f1_score(y_true_r, ensemble_pred_r, average='weighted', zero_division=0)*100:.2f}%")
print("\n=== Per-class Report ===")
print(classification_report(y_true_r, ensemble_pred_r,
      target_names=keep_names, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_true_r, ensemble_pred_r)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=keep_names, yticklabels=keep_names)
plt.title('Real Test — Ensemble Confusion Matrix (ISIC 2019)')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_realtest_ensemble.png', dpi=150)
plt.close()
print("\n✅ 已儲存 confusion_realtest_ensemble.png")

✅ best_mobilenet_ft.h5 載入成功
✅ best_densenet_ft.h5 載入成功
✅ best_efficientnet_ft.h5 載入成功

預測中...
6/6 [==============================] - 5s 200ms/step

=== Real Test Results (ISIC 2019 — 6 classes) ===
Accuracy:    78.92%
Macro F1:    56.01%
Weighted F1: 80.31%

=== Per-class Report ===
              precision    recall  f1-score   support

         bcc       0.79      0.73      0.76        15
         bkl       0.54      0.91      0.68        22
          df       0.00      0.00      0.00         1
         mel       0.50      0.57      0.53        21
          nv       0.96      0.83      0.89       123
        vasc       1.00      0.33      0.50         3

    accuracy                           0.79       185
   macro avg       0.63      0.56      0.56       185
weighted avg       0.84      0.79      0.80       185


✅ 已儲存 confusion_realtest_ensemble.png


In [22]:
# ============================================================
# STEP 8: Application Flow 流程圖
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

def box(ax, x, y, w, h, text, color, fontsize=10):
    fancy = FancyBboxPatch((x - w/2, y - h/2), w, h,
                           boxstyle="round,pad=0.15",
                           facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(fancy)
    ax.text(x, y, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white',
            multialignment='center')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#444444',
                                lw=2.0, connectionstyle='arc3,rad=0'))

# 節點
box(ax,  2,   6.2, 2.4, 1.2, "Patient\nUploads Image",            "#2196F3")
box(ax,  5,   6.2, 2.4, 1.2, "Preprocessing\n224x224 / Normalize","#1976D2")
box(ax,  8,   6.2, 2.6, 1.3, "Ensemble Model\nMobileNet+DenseNet\n+EfficientNetB3", "#1565C0")
box(ax, 11.2, 6.2, 2.4, 1.2, "Prediction\n7 Classes",             "#0D47A1")
box(ax, 11.2, 3.8, 2.4, 1.2, "High Risk?\nMel / BCC / AK",        "#C62828")
box(ax,  8,   2.0, 2.4, 1.0, "Refer to\nDermatologist",            "#B71C1C")
box(ax, 11.2, 2.0, 2.4, 1.0, "Low Risk\nMonitor / Reassure",       "#2E7D32")

# 箭頭
arrow(ax,  3.2,  6.2,  3.8,  6.2)
arrow(ax,  6.2,  6.2,  6.7,  6.2)
arrow(ax,  9.3,  6.2,  9.95, 6.2)
arrow(ax, 11.2,  5.6, 11.2,  4.4)
arrow(ax, 10.0,  3.4,  9.2,  2.5)
arrow(ax, 11.2,  3.2, 11.2,  2.5)

# 標題
ax.text(7, 7.5, 'AI-Assisted Skin Lesion Screening — Application Flow',
        ha='center', va='center', fontsize=14, fontweight='bold', color='#1A237E')

# 結果標註框
ax.text(5.5, 4.0,
        'Model Performance Summary\n'
        'HAM10000 Test:  Acc 76.51%  |  Macro F1 64.08%\n'
        'ISIC 2019 Real: Acc 78.92%  |  Macro F1 56.01%',
        ha='center', va='center', fontsize=9, color='#333333',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#F0F4FF', edgecolor='#90CAF9', linewidth=1.5))

# SDG 標註
ax.text(5.5, 2.2,
        'SDG 3.4 — Reduce premature mortality\nSDG 3.8 — Universal health coverage',
        ha='center', va='center', fontsize=9, color='#1B5E20',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#F1F8E9', edgecolor='#A5D6A7', linewidth=1.5))

plt.tight_layout()
plt.savefig('application_flow.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ 已儲存 application_flow.png")

✅ 已儲存 application_flow.png


In [30]:
# ============================================================
# STEP 9: ROC AUC 數字表格
# ============================================================
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
import pandas as pd
import numpy as np

CLASS_NAMES_EVAL = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
y_true = test_gen.classes
y_bin  = label_binarize(y_true, classes=range(NUM_CLASSES))

ensemble_prob = (m_prob + d_prob + e_prob) / 3

auc_results = {}
for model_name, prob in [('MobileNetV2',    m_prob),
                          ('DenseNet121',    d_prob),
                          ('EfficientNetB3', e_prob),
                          ('Ensemble',       ensemble_prob)]:
    aucs = []
    for i in range(NUM_CLASSES):
        auc = roc_auc_score(y_bin[:, i], prob[:, i])
        aucs.append(round(auc, 3))
    auc_results[model_name] = aucs

auc_df = pd.DataFrame(auc_results, index=CLASS_NAMES_EVAL)
auc_df['Mean AUC'] = auc_df.mean(axis=1).round(3)
auc_df.loc['Mean'] = auc_df.mean().round(3)

print("=== ROC AUC per Class ===")
print(auc_df.to_string())

=== ROC AUC per Class ===
       MobileNetV2  DenseNet121  EfficientNetB3  Ensemble  Mean AUC
akiec        0.926        0.936           0.925     0.965     0.938
bcc          0.937        0.931           0.959     0.961     0.947
bkl          0.875        0.890           0.886     0.925     0.894
df           0.893        0.957           0.926     0.974     0.938
mel          0.857        0.776           0.813     0.866     0.828
nv           0.921        0.928           0.919     0.949     0.929
vasc         0.903        0.938           0.979     0.979     0.950
Mean         0.902        0.908           0.915     0.946     0.918


In [31]:
# ============================================================
# STEP 10: 專業圖表生成
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# ── 圖1: Model Performance Comparison（橫條圖）──────────────
models = ['MobileNetV2', 'DenseNet121', 'EfficientNetB3', 'Ensemble\n(3 models)']
accuracy  = [68.53, 64.87, 73.65, 76.51]
macro_f1  = [54.97, 51.70, 56.88, 64.08]
weighted_f1 = [71.26, 68.23, 74.78, 77.95]

x = np.arange(len(models))
w = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - w, accuracy,    w, label='Accuracy (%)',    color=colors[0], alpha=0.85)
b2 = ax.bar(x,     macro_f1,    w, label='Macro F1 (%)',    color=colors[1], alpha=0.85)
b3 = ax.bar(x + w, weighted_f1, w, label='Weighted F1 (%)', color=colors[2], alpha=0.85)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_ylim(0, 90)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='lower right')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axvline(x=2.5, color='#CCCCCC', linestyle='--', linewidth=1)
ax.text(2.6, 83, 'Ensemble', fontsize=9, color='#888888', style='italic')
plt.tight_layout()
plt.savefig('chart_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ chart_model_comparison.png")

# ── 圖2: ROC AUC Heatmap ─────────────────────────────────────
auc_data = pd.DataFrame({
    'MobileNetV2':   [0.926, 0.937, 0.875, 0.893, 0.857, 0.921, 0.903],
    'DenseNet121':   [0.936, 0.931, 0.890, 0.957, 0.776, 0.928, 0.938],
    'EfficientNetB3':[0.925, 0.959, 0.886, 0.926, 0.813, 0.919, 0.979],
    'Ensemble':      [0.965, 0.961, 0.925, 0.974, 0.866, 0.949, 0.979],
}, index=['akiec','bcc','bkl','df','mel','nv','vasc'])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(auc_data, annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0.75, vmax=1.0, linewidths=0.5, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'}, ax=ax)
ax.set_title('ROC AUC Scores by Class and Model', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Class', fontsize=12)
ax.tick_params(axis='x', rotation=15)
ax.tick_params(axis='y', rotation=0)

# 加 Mean row annotation
mean_vals = auc_data.mean()
for j, val in enumerate(mean_vals):
    ax.text(j + 0.5, 7.3, f'μ={val:.3f}', ha='center', va='center',
            fontsize=9, color='#333333', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_roc_auc_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ chart_roc_auc_heatmap.png")

# ── 圖3: Focal Loss vs CE 對比（雷達圖）─────────────────────
from matplotlib.patches import FancyArrowPatch

categories = ['akiec','bcc','bkl','df','mel','nv','vasc']
N = len(categories)

ce_recall   = [0.37, 0.49, 0.62, 0.59, 0.34, 0.70, 0.82]  # EfficientNetB3 原版 recall
fl_recall   = [0.41, 0.64, 0.77, 0.47, 0.39, 0.77, 0.82]  # Focal Loss recall

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
ce_recall   += ce_recall[:1]
fl_recall   += fl_recall[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, ce_recall, 'o-', linewidth=2, color='#2196F3', label='Cross-Entropy')
ax.fill(angles, ce_recall, alpha=0.15, color='#2196F3')
ax.plot(angles, fl_recall, 's-', linewidth=2, color='#FF5722', label='Focal Loss')
ax.fill(angles, fl_recall, alpha=0.15, color='#FF5722')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8, color='grey')
ax.set_title('Per-class Recall: Cross-Entropy vs Focal Loss\n(EfficientNetB3)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.savefig('chart_focal_vs_ce_radar.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ chart_focal_vs_ce_radar.png")

# ── 圖4: Real Test vs HAM10000 比較 ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左：Accuracy 比較
datasets = ['HAM10000\nTest Set', 'ISIC 2019\nReal Test']
acc_vals = [76.51, 78.92]
colors_bar = ['#1565C0', '#2E7D32']
bars = axes[0].bar(datasets, acc_vals, color=colors_bar, width=0.4, alpha=0.85)
for bar, val in zip(bars, acc_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 95)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Ensemble Accuracy\nHAM10000 vs ISIC 2019', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3, linestyle='--')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# 右：全指標比較
metrics = ['Accuracy', 'Macro F1', 'Weighted F1']
ham_vals  = [76.51, 64.08, 77.95]
isic_vals = [78.92, 56.01, 80.31]
x2 = np.arange(len(metrics))
w2 = 0.3
b1 = axes[1].bar(x2 - w2/2, ham_vals,  w2, label='HAM10000',   color='#1565C0', alpha=0.85)
b2 = axes[1].bar(x2 + w2/2, isic_vals, w2, label='ISIC 2019',  color='#2E7D32', alpha=0.85)
for bars2 in [b1, b2]:
    for bar in bars2:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(metrics, fontsize=11)
axes[1].set_ylim(0, 95)
axes[1].set_ylabel('Score (%)', fontsize=12)
axes[1].set_title('All Metrics Comparison\nHAM10000 vs ISIC 2019', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3, linestyle='--')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Ensemble Model — Generalization to External Dataset', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_realtest_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ chart_realtest_comparison.png")

print("\n全部圖表生成完成！")

✅ chart_model_comparison.png
✅ chart_roc_auc_heatmap.png
✅ chart_focal_vs_ce_radar.png
✅ chart_realtest_comparison.png

全部圖表生成完成！
